### Imports

In [1]:
import torch
from torchvision.models import resnet18, ResNet18_Weights
import torch.nn as nn
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os
from torchvision.transforms import transforms

import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.metrics import accuracy_score, classification_report

### Load ResNet18 Model + GRU

In [ ]:
class Experiment3(nn.Module):
    def __init__(self, num_classes=4):
        super(Experiment3, self).__init__()
        # load pretrained model
        resnet_model = resnet18(weights=ResNet18_Weights.DEFAULT)

        # remove last fully connected layer
        self.feature_extractor = nn.Sequential(*list(resnet_model.children())[:-1])

        # define GRU
        self.gru = nn.GRU(input_size=512, hidden_size=128, batch_first=True) # hidden_size möglicherweise anpassen

        # final classifier
        self.fc = nn.Linear(128, num_classes) 


    def forward(self,x):
        batch_size, seq_len, channels, height, width = x.size()

        # use ResNet for each picture in the sequence
        x = x.view(batch_size * seq_len, channels, height, width)
        with torch.no_grad(): # ResNet eingefroren
            features = self.feature_extractor(x)

        # bring features into right sequential form
        features = torch.flatten(features, 1)
        features = features.view(batch_size, seq_len, -1)

        gru_out, hn = self.gru(features)

        last_hidden_state = hn[-1]

        output = self.fc(last_hidden_state)
        return output

### Load Dataset

In [5]:
# define path 
DATASET_PATH = "/Users/gwen/Desktop/video_frames_data"
CSV_PATH = os.path.join(DATASET_PATH, "metadata.csv")

# load dataset
class VideoDataset(Dataset):
    def __init__(self, dataframe, root_dir, transform=None):
        self.df = dataframe 
        self.root_dir = root_dir
        self.transform = transform
        self.seq_len = 8 

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        video_folder = os.path.join(self.root_dir, row['frame_dir'] )

        video_class = int(row['class_id']) 

        img_sequence = sorted(os.listdir(video_folder))


        frames = []
        for img_name in img_sequence:
            img_path = os.path.join(video_folder, img_name)
            image = Image.open(img_path).convert('RGB') # to make 100% sure
            if self.transform:
                image = self.transform(image)
            frames.append(image)

        return torch.stack(frames), video_class


# define transformations for our dataset
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])

# load full csv
full_df = pd.read_csv(CSV_PATH)

full_df['frame_dir'] = full_df['frame_dir'].str.replace('\\', '/', regex=False) # wegen Windows und Mac

# load training data
train_df = full_df[full_df['split'] == 'train'].reset_index(drop=True)
# load validation data
val_df = full_df[full_df['split'] == 'val'].reset_index(drop=True)

# Dataset and Loader
train_dataset = VideoDataset(dataframe=train_df, root_dir=DATASET_PATH, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_dataset = VideoDataset(dataframe=val_df, root_dir=DATASET_PATH, transform=transform)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)


### Training Loop

In [4]:
# training loop 
def train(model, train_loader, val_loader, criterion, optimizer, num_epochs):
    # determine device
    device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")

    for epoch in range(num_epochs):
        # training mode
        model.train()

        running_loss = 0.0
        running_corrects = 0

        # iterate training data loader
        for inputs, labels in train_loader:
            # put to device
            inputs = inputs.to(device)
            labels = labels.to(device)

            # reset gradients 
            optimizer.zero_grad()

            # forward pass
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            loss = criterion(outputs, labels)

            # backward pass
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

        # epoch training loss and accuracy 
        train_loss = running_loss / len(train_loader.dataset)
        train_acc = running_corrects.float() / len(train_loader.dataset)


        ## Evaluation
        model.eval()
        running_loss = 0.0
        running_corrects = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs = inputs.to(device)
                labels = labels.to(device)

                # forward pass
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

        # validation loss and accuracy 
        val_loss = running_loss / len(val_loader.dataset)
        val_acc = running_corrects.float() / len(val_loader.dataset)

        print(f'Epoch [{epoch+1}/{num_epochs}], train loss: {train_loss:.4f}, train acc: {train_acc:.4f}, val loss: {val_loss:.4f}, val acc: {val_acc:.4f} \n')

In [5]:
# train model
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
model = Experiment3(num_classes=4).to(device)
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001) 

train(model, train_loader, val_loader, criterion, optimizer, num_epochs=20)

# save checkpoint of model
checkpoint = {
    "Experiment3_state_dict": model.state_dict(),
    "optimizer_Experiment3_state_dict": optimizer.state_dict(),
    "epoch": 20,
}

torch.save(
    checkpoint,
    "checkpoints/Experiment3_checkpoint.pth"
)

Epoch [1/20], train loss: 1.0245, train acc: 0.4682, val loss: 0.9230, val acc: 0.5158 

Epoch [2/20], train loss: 0.8534, train acc: 0.5509, val loss: 0.8199, val acc: 0.5702 

Epoch [3/20], train loss: 0.8096, train acc: 0.5968, val loss: 0.7945, val acc: 0.5842 

Epoch [4/20], train loss: 0.7692, train acc: 0.6174, val loss: 0.7718, val acc: 0.5947 

Epoch [5/20], train loss: 0.7459, train acc: 0.6298, val loss: 0.7652, val acc: 0.5825 

Epoch [6/20], train loss: 0.7321, train acc: 0.6576, val loss: 0.7857, val acc: 0.6018 

Epoch [7/20], train loss: 0.7300, train acc: 0.6400, val loss: 0.7631, val acc: 0.6070 

Epoch [8/20], train loss: 0.6723, train acc: 0.6772, val loss: 0.7729, val acc: 0.6105 

Epoch [9/20], train loss: 0.6732, train acc: 0.6719, val loss: 0.7784, val acc: 0.5860 

Epoch [10/20], train loss: 0.6507, train acc: 0.7005, val loss: 0.8537, val acc: 0.5860 

Epoch [11/20], train loss: 0.7146, train acc: 0.6776, val loss: 0.7499, val acc: 0.6439 

Epoch [12/20], trai

### Evaluation 

In [7]:

def evaluate_Experiment3(model, loader, device):
    model.eval()

    true_labels = []
    preds = []
    
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            true_labels.extend(labels.cpu().numpy())
            preds.extend(predicted.cpu().numpy())

    print(f"Overall Accuracy: {accuracy_score(true_labels, preds):.4f}")
    print(classification_report(true_labels, preds))
    
    report = classification_report(true_labels, preds, output_dict=True)
    return accuracy_score(true_labels, preds), {f"class{i+1}": report[str(i)]["recall"] for i in range(4) if str(i) in report}

device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")

# B. Modell laden
model = Experiment3(num_classes=4).to(device)
checkpoint = torch.load("checkpoints/Experiment3_checkpoint.pth", map_location=device)
model.load_state_dict(checkpoint["Experiment3_state_dict"])

# C. Evaluieren
acc_Experiment3, acc_classes_Experiment3 = evaluate_Experiment3(model, val_loader, device)

Overall Accuracy: 0.6211
              precision    recall  f1-score   support

           0       1.00      0.99      1.00       144
           1       0.44      0.47      0.45       142
           2       0.68      0.35      0.46       142
           3       0.47      0.67      0.55       142

    accuracy                           0.62       570
   macro avg       0.65      0.62      0.62       570
weighted avg       0.65      0.62      0.62       570



### Load ResNet18 Model + GRU and unfreeze ResNet18

In [2]:
class model_gru_layer4(nn.Module):
    def __init__(self, num_classes=4):
        super(model_gru_layer4, self).__init__()
        # load pretrained model
        resnet_model = resnet18(weights=ResNet18_Weights.DEFAULT)

        # remove last fully connected layer
        self.feature_extractor = nn.Sequential(*list(resnet_model.children())[:-1])

        # define GRU
        self.gru = nn.GRU(input_size=512, hidden_size=128, batch_first=True) # hidden_size möglicherweise anpassen

        # final classifier
        self.fc = nn.Linear(128, num_classes) 


    def forward(self,x):
        batch_size, seq_len, channels, height, width = x.size()

        # use ResNet for each picture in the sequence
        x = x.view(batch_size * seq_len, channels, height, width)
        
        features = self.feature_extractor(x)

        # bring features into right sequential form
        features = torch.flatten(features, 1)
        features = features.view(batch_size, seq_len, -1)

        gru_out, hn = self.gru(features)

        last_hidden_state = hn[-1]

        output = self.fc(last_hidden_state)
        return output

### Load Dataset for unfreezed model

In [3]:
# define path 
DATASET_PATH = "/Users/gwen/Desktop/video_frames_data"
CSV_PATH = os.path.join(DATASET_PATH, "metadata.csv")

# load dataset
class VideoDataset(Dataset):
    def __init__(self, dataframe, root_dir, transform=None):
        self.df = dataframe 
        self.root_dir = root_dir
        self.transform = transform
        self.seq_len = 8 

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        video_folder = os.path.join(self.root_dir, row['frame_dir'] )

        video_class = int(row['class_id']) 

        img_sequence = sorted(os.listdir(video_folder))


        frames = []
        for img_name in img_sequence:
            img_path = os.path.join(video_folder, img_name)
            image = Image.open(img_path).convert('RGB') # to make 100% sure
            if self.transform:
                image = self.transform(image)
            frames.append(image)

        return torch.stack(frames), video_class


# define transformations for our dataset
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])

# load full csv
full_df = pd.read_csv(CSV_PATH)

full_df['frame_dir'] = full_df['frame_dir'].str.replace('\\', '/', regex=False) # wegen Windows und Mac

# load training data
train_df = full_df[full_df['split'] == 'train'].reset_index(drop=True)
# load validation data
val_df = full_df[full_df['split'] == 'val'].reset_index(drop=True)

# Dataset and Loader
train_dataset = VideoDataset(dataframe=train_df, root_dir=DATASET_PATH, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_dataset = VideoDataset(dataframe=val_df, root_dir=DATASET_PATH, transform=transform)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)


### Training Loop for unfreesed model 

In [4]:
# training loop 
def train(model, train_loader, val_loader, criterion, optimizer, num_epochs):
    # determine device
    device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")

    for epoch in range(num_epochs):
        # training mode
        model.train()

        running_loss = 0.0
        running_corrects = 0

        # iterate training data loader
        for inputs, labels in train_loader:
            # put to device
            inputs = inputs.to(device)
            labels = labels.to(device)

            # reset gradients 
            optimizer.zero_grad()

            # forward pass
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            loss = criterion(outputs, labels)

            # backward pass
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

        # epoch training loss and accuracy 
        train_loss = running_loss / len(train_loader.dataset)
        train_acc = running_corrects.float() / len(train_loader.dataset)


        ## Evaluation
        model.eval()
        running_loss = 0.0
        running_corrects = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs = inputs.to(device)
                labels = labels.to(device)

                # forward pass
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

        # validation loss and accuracy 
        val_loss = running_loss / len(val_loader.dataset)
        val_acc = running_corrects.float() / len(val_loader.dataset)

        print(f'Epoch [{epoch+1}/{num_epochs}], train loss: {train_loss:.4f}, train acc: {train_acc:.4f}, val loss: {val_loss:.4f}, val acc: {val_acc:.4f} \n')

In [5]:
# train model
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
model = model_gru_layer4(num_classes=4).to(device)

# Freeze all layers
for param in model.feature_extractor.parameters():
    param.requires_grad = False

# Unfreeze layer 4
for param in model.feature_extractor[7].parameters():
    param.requires_grad = True

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam([
    {"params": model.feature_extractor[7].parameters(), "lr": 1e-5},
    {"params": list(model.gru.parameters()) + list(model.fc.parameters()), "lr": 0.001},
], weight_decay=0.006)

train(model, train_loader, val_loader, criterion, optimizer, num_epochs=20)

# save checkpoint of model
checkpoint = {
    "model_gru_layer4_state_dict": model.state_dict(),
    "optimizer_model_gru_layer4_state_dict": optimizer.state_dict(),
    "epoch": 20,
}

torch.save(
    checkpoint,
    "checkpoints/model_gru_layer4_checkpoint.pth"
)

Epoch [1/20], train loss: 0.9716, train acc: 0.4863, val loss: 0.8538, val acc: 0.5018 

Epoch [2/20], train loss: 0.7767, train acc: 0.6002, val loss: 0.7617, val acc: 0.6175 

Epoch [3/20], train loss: 0.7017, train acc: 0.6614, val loss: 0.7308, val acc: 0.6421 

Epoch [4/20], train loss: 0.6238, train acc: 0.7208, val loss: 0.6770, val acc: 0.6754 

Epoch [5/20], train loss: 0.4913, train acc: 0.7948, val loss: 0.6740, val acc: 0.6965 

Epoch [6/20], train loss: 0.3826, train acc: 0.8482, val loss: 0.7153, val acc: 0.6877 

Epoch [7/20], train loss: 0.2948, train acc: 0.8801, val loss: 0.7853, val acc: 0.6860 

Epoch [8/20], train loss: 0.2228, train acc: 0.9169, val loss: 0.7978, val acc: 0.7035 

Epoch [9/20], train loss: 0.1761, train acc: 0.9357, val loss: 0.9713, val acc: 0.6772 

Epoch [10/20], train loss: 0.1287, train acc: 0.9542, val loss: 0.9259, val acc: 0.6947 

Epoch [11/20], train loss: 0.1191, train acc: 0.9620, val loss: 1.0847, val acc: 0.6772 

Epoch [12/20], trai